# Validity checks and per-attack table
Runtime > Run all. Checks that DReLAB adversarial flows have integer counts and satisfy TotBytes = InBytes + OutBytes and BytesPerPkt = TotBytes / TotPkts, as the original malware does (paper Section 5.7), and prints the per-attack table (Table 5).

In [ ]:
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
os.makedirs(OUT, exist_ok=True)
FAM = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"], "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"], "CICIDS2018": ["Zeus_Ares"]}
ATT = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]


In [ ]:
# Validity checks (paper Section 5.7): are the adversarial flows internally consistent, like the original malware?
COUNT = ["TotPkts", "InBytes", "OutBytes", "TotBytes"]
def load(ds, kind):
    paths = ([f"{BASE}/{ds}/malicious/{f}.csv" for f in FAM[ds]] if kind == "orig" else [f"{BASE}/{ds}/{kind[0]}/{kind[1]}/{f}.csv" for f in FAM[ds]])
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
rows = []
for ds in FAM:
    for kind in ["orig"] + ATT:
        X = load(ds, kind); r = dict(dataset=ds, flows="original" if kind == "orig" else "/".join(kind), n=len(X))
        for c in COUNT:
            if c in X: r["non-integer " + c] = round(float((np.abs(X[c] - np.round(X[c])) > 1e-6).mean()), 4)
        if all(c in X for c in ["TotBytes", "InBytes", "OutBytes"]):
            r["TotBytes != In+Out"] = round(float((np.abs(X["TotBytes"] - X["InBytes"] - X["OutBytes"]) > 1).mean()), 4)
        if all(c in X for c in ["BytesPerPkt", "TotBytes", "TotPkts"]):
            r["BytesPerPkt != Tot/Pkts"] = round(float((~np.isclose(X["BytesPerPkt"], X["TotBytes"] / np.maximum(X["TotPkts"], 1), rtol=0.02, atol=1)).mean()), 4)
        rows.append(r)
chk = pd.DataFrame(rows); chk.to_csv(f"{OUT}/validity_checks.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", 20); print(chk.to_string(index=False))


In [ ]:
# Per-attack results of the ensemble detector (paper Table 5), from the CSV written by revision_full_run.ipynb
path = f"{OUT}/per_seed_results.csv"
if os.path.exists(path):
    res = pd.read_csv(path); r = res[(res.method == "Ensemble disagreement only") & (res.attack != "ALL")]
    pm = lambda x: f"{np.mean(x):.3f} +/- {np.std(x, ddof=1):.3f}"
    print(r.groupby(["dataset", "attack"])[["AUROC", "TPR", "FPR"]].agg(pm).reset_index().to_string(index=False))
else:
    print("Run revision_full_run.ipynb first; it writes per_seed_results.csv.")
